# Axiom `.axim` SFT - one-click Colab

Fine-tune an **Axiom / nanoChat** model (packaged as `.axim`) on a **reasoning + chat** dataset, end to end in Colab. Each cell below is **one command** - run them top to bottom, or Run All.

1. Configure everything in the **CONFIG** cell.
2. The next cell auto-tunes precision / batch size for your specific GPU.
3. Install + clone repos.
4. Download the base model from the HuggingFace Hub (the repo ships a ready `.axim`).
5. Prepare a reasoning+chat dataset into train/val JSONL.
6. Fine-tune -> writes a new `.axim` that's a drop-in replacement for the base.
7. Generate from the fine-tuned model to eyeball it, then download it.

> **Runtime -> Change runtime type -> GPU** (T4 works but slow; A100/L4 is ~10x faster). A 1.38B model fits on a free T4 with the auto-chosen settings.


## 1. CONFIG - edit this cell

Plain Python dict. The auto-tuner next cell reads it.

In [ ]:
#@title CONFIG  { run: "auto" }
CONFIG = {
    # base model on the Hub (this repo ships a ready .axim)
    "HF_MODEL_REPO": "StrawberCar/Axiom-V1-Base",

    # ---- backbone (general chat + talkability) ----
    # smol-smoltalk = nanoChat's OWN conversational backbone for this model class
    # (460K rows, multi-turn chat + persona roleplay). DATASET_CONFIG=None -> default config.
    "DATASET_REPO":   "HuggingFaceTB/smol-smoltalk",
    "DATASET_CONFIG": None,
    "TRAIN_EXAMPLES": 50000,
    "VAL_EXAMPLES":   200,

    # ---- personality blend (automated, mixed into the same training run) ----
    # A persona dataset blended on top of the backbone so the model "learns a
    # personality" in ONE run, with no second fine-tune step. The backbone keeps
    # it capable/talkable; this slice adds flavor. PERSONA_REPEAT copies it into
    # the train file that many times (~3-5% of the mix = a safe, noticeable spike).
    # Set PERSONA_REPO=None to disable and train backbone-only.
    #   anime (SFW, drop-in): "zerofata/Roleplay-Anime-Characters"
    #   charcard roleplay:   "Gryphe/Sonnet3.5-Charcard-Roleplay"
    "PERSONA_REPO":   "zerofata/Roleplay-Anime-Characters",
    "PERSONA_CONFIG": None,
    "PERSONA_REPEAT": 5,

    # training (base -> chat SFT). EPOCHS=2 = enough to lock the chat protocol
    # without the epoch-3 overfit risk; the WSD warmdown (last 20%) still anneals
    # it to a clean minimum. Set 3 for max learning, 1 for fastest.
    "OUTPUT_NAME":  "finetuned",     # -> /content/axim_work/finetuned.axim
    "EPOCHS":       2,
    "TRAIN_ON":     "assistant",     # "assistant" = supervise only answers; "all" = also prompts
    "PACK":         "bestfit",       # "bestfit" = pack many convs/row (efficient); "single" = 1/row
    "SYSTEM_PROMPT": None,           # str -> prepend as system msg to records lacking one
    "SEED": 42,

    # cadence (steps)
    "EVAL_EVERY":   50,
    "SAMPLE_EVERY": 100,
    "LOG_EVERY":    1,
    # save a full .axim snapshot every N steps so a Colab disconnect (or early
    # stop) doesn't lose the run. ~2.8GB each; 500 over ~2700 steps ~ 14GB.
    "SAVE_EVERY":   500,

    # ---- in-training generation samples (eyeball quality as it learns) ----
    # SAMPLE_TOKENS: max tokens per sample generation. Higher = fair eval of
    #   outputs that don't emit the stop token (a "[no-stop]" flag marks those so
    #   you can see when the model rambles instead of stopping).
    # SAMPLE_STATIC_PROMPTS: FIXED prompts generated at EVERY sample-every, so
    #   you can watch the model improve on identical inputs (controlled env).
    #   Edit these to whatever you want to track (chat / anime / creative).
    # SAMPLE_RANDOM: how many RANDOM val prompts to ALSO generate each sample
    #   (re-drawn every time, seeded by step, for variety / generalization checks).
    "SAMPLE_TOKENS":         256,
    "SAMPLE_RANDOM":         3,
    "SAMPLE_STATIC_PROMPTS": [
        "Hi! Who are you?",
        "What do you like to do for fun?",
        "Tell me a short story about a lost kitten.",
    ],

    # ---- A100 throughput overrides (the auto-tuner's batch=4 leaves the GPU
    # loafing). MICRO_BATCH x GRAD_ACCUM is kept at 16 x 2048 = 32768 tokens/step
    # (SAME total batch as before, so LR / optimizer dynamics are UNCHANGED);
    # a bigger micro-batch just fills the A100 better -> ~1.5-2x faster.
    # If you OOM at startup, set MICRO_BATCH_OVERRIDE=4 and GRAD_ACCUM_OVERRIDE=4
    # (the original safe values) and you still keep the compile + epochs speedup.
    "MICRO_BATCH_OVERRIDE": 8,
    "GRAD_ACCUM_OVERRIDE":  2,
    # torch.compile the model (Karpathy's nanochat is built for this) -> another
    # ~1.3-1.5x. The first few steps are slow while it compiles. If it errors,
    # set COMPILE=0 (you keep the batch + epochs speedup).
    "COMPILE":               1,

    # fine-tuning learning rates (Muon). ~1/5 of pretraining scale = gentle SFT.
    # We can NOT warm-start the optimizer (nanochat does; .axim has no optimizer
    # state), so high LR thrashes the good base minimum -> loss climbs. Keep low.
    # If loss still rises, lower these; if training too slow/flat, raise slightly.
    "MATRIX_LR":      0.004,
    "EMBEDDING_LR":   0.06,
    "UNEMBEDDING_LR": 0.0008,
    "SCALAR_LR":      0.1,
    "WARMUP_RATIO":   0.05,
    "INIT_LR_FRAC":   0.5,

    # optional overrides (None = let the auto-tuner pick for your GPU)
    "MAX_SEQ_LEN_OVERRIDE": None,
    "OPTIMIZER_OVERRIDE":   None,   # "muon" | "adamw"
}
print("CONFIG loaded.")

## 2. Detect GPU + auto-tune precision / batch

The base is stored at full precision. On a free **T4 (SM7x, no bf16 compute)** we cast to fp16 and use small batches; on **Ampere+ (A100/L4/H100, SM8+)** we keep bf16 and use bigger batches. All paths are set up as plain variables used by the command cells below.

In [ ]:
import torch, os, shlex, json, subprocess
assert torch.cuda.is_available(), "No GPU! In Colab: Runtime > Change runtime type > T4 (or A100) GPU."
major = torch.cuda.get_device_capability()[0]
print(f"GPU: {torch.cuda.get_device_name()}  |  SM{major}")

if major >= 8:        # A100 / L4 / H100 - bf16 native (base is fp32 -> cast to bf16 for 2x speed)
    DTYPE, CAST, BATCH, GRAD_ACCUM, SEQ, OPT = "bfloat16", 1, 4, 4, 2048, "muon"
elif major == 7:      # T4 / V100 - no bf16 compute
    DTYPE, CAST, BATCH, GRAD_ACCUM, SEQ, OPT = "float16", 1, 1, 16, 1024, "muon"
else:                 # older
    DTYPE, CAST, BATCH, GRAD_ACCUM, SEQ, OPT = "float16", 1, 1, 32, 512, "muon"
if CONFIG["MAX_SEQ_LEN_OVERRIDE"]: SEQ = CONFIG["MAX_SEQ_LEN_OVERRIDE"]
if CONFIG["OPTIMIZER_OVERRIDE"]:   OPT = CONFIG["OPTIMIZER_OVERRIDE"]
# A100 throughput overrides (see CONFIG). Total batch = BATCH*GRAD_ACCUM*SEQ is
# kept CONSTANT, so LR / optimizer dynamics are unchanged; a bigger micro-batch
# just fills the GPU better. Fallback to 4/4 if you OOM at startup.
if CONFIG.get("MICRO_BATCH_OVERRIDE"): BATCH = CONFIG["MICRO_BATCH_OVERRIDE"]
if CONFIG.get("GRAD_ACCUM_OVERRIDE"):  GRAD_ACCUM = CONFIG["GRAD_ACCUM_OVERRIDE"]
COMPILE = CONFIG.get("COMPILE", 0)

# paths + derived vars used by the command cells
WORK        = "/content/axim_work"
AXIM_REPO   = f"{WORK}/Axiom-Inference-Engine"
NANO_REPO   = f"{AXIM_REPO}/nanochat"
os.makedirs(WORK, exist_ok=True)   # ensure WORK exists before ANY file write / git op below

# Sync the trainer to latest origin/main so a RESTARTED run (reused runtime,
# skipping the clone cell) still picks up repo changes such as the static +
# random sample feature in sft_train.py. No-op on a fresh runtime where the repo
# hasn't been cloned yet (the clone cell below grabs it fresh).
if os.path.isdir(os.path.join(AXIM_REPO, ".git")):
    try:
        subprocess.run(["git", "-C", AXIM_REPO, "fetch", "--depth", "1", "origin", "main"],
                       check=False, capture_output=True)
        subprocess.run(["git", "-C", AXIM_REPO, "reset", "--hard", "origin/main"],
                       check=False, capture_output=True)
        print(f"synced trainer to latest origin/main -> {AXIM_REPO}")
    except Exception as _e:
        print(f"  (trainer sync skipped: {_e})")

HF_REPO     = CONFIG["HF_MODEL_REPO"]
BASE_AXIM   = f"{WORK}/base.axim"
DATASET_REPO, DATASET_CONFIG = CONFIG["DATASET_REPO"], CONFIG["DATASET_CONFIG"]
TRAIN_EXAMPLES, VAL_EXAMPLES = CONFIG["TRAIN_EXAMPLES"], CONFIG["VAL_EXAMPLES"]
TRAIN_JSONL, VAL_JSONL = f"{WORK}/train.jsonl", f"{WORK}/val.jsonl"
OUT_AXIM    = f"{WORK}/{CONFIG['OUTPUT_NAME']}.axim"
OUT_DIR     = f"{WORK}/sft_checkpoints"
EPOCHS, TRAIN_ON, PACK = CONFIG["EPOCHS"], CONFIG["TRAIN_ON"], CONFIG["PACK"]
SEED, SYSTEM_PROMPT   = CONFIG["SEED"], CONFIG["SYSTEM_PROMPT"]
PERSONA_REPO, PERSONA_CONFIG, PERSONA_REPEAT = CONFIG["PERSONA_REPO"], CONFIG["PERSONA_CONFIG"], CONFIG["PERSONA_REPEAT"]
PERSONA_CFGARG = f"--config {PERSONA_CONFIG}" if PERSONA_CONFIG else ""
EVAL_EVERY, SAMPLE_EVERY, SAVE_EVERY, LOG_EVERY = CONFIG["EVAL_EVERY"], CONFIG["SAMPLE_EVERY"], CONFIG["SAVE_EVERY"], CONFIG["LOG_EVERY"]
MATRIX_LR, EMBEDDING_LR, UNEMBEDDING_LR, SCALAR_LR = CONFIG["MATRIX_LR"], CONFIG["EMBEDDING_LR"], CONFIG["UNEMBEDDING_LR"], CONFIG["SCALAR_LR"]
WARMUP_RATIO, INIT_LR_FRAC = CONFIG["WARMUP_RATIO"], CONFIG["INIT_LR_FRAC"]
CFGARG = f"--config {DATASET_CONFIG}" if DATASET_CONFIG else ""
SYSARG = ("--system-prompt " + shlex.quote(SYSTEM_PROMPT)) if SYSTEM_PROMPT else ""

# in-training sample settings (see CONFIG). Write the FIXED prompts to a JSONL
# file the trainer loads once and reuses every sample-every; the random ones are
# drawn fresh inside the trainer (seeded by step).
SAMPLE_TOKENS         = CONFIG.get("SAMPLE_TOKENS", 256)
SAMPLE_RANDOM         = CONFIG.get("SAMPLE_RANDOM", 3)
SAMPLE_STATIC_PROMPTS = CONFIG.get("SAMPLE_STATIC_PROMPTS", []) or []
SAMPLE_STATIC_FILE    = f"{WORK}/sample_static.jsonl"
with open(SAMPLE_STATIC_FILE, "w", encoding="utf-8") as _f:
    for _p in SAMPLE_STATIC_PROMPTS:
        _f.write(json.dumps({"messages": [{"role": "user", "content": str(_p)}]}, ensure_ascii=False) + "\n")
print(f"static sample prompts: {len(SAMPLE_STATIC_PROMPTS)} -> {SAMPLE_STATIC_FILE}")

os.environ["NANOCHAT_DTYPE"] = DTYPE
print(f"plan: dtype={DTYPE} cast={CAST} batch={BATCH}x{GRAD_ACCUM} seq={SEQ} opt={OPT} compile={COMPILE} save_every={SAVE_EVERY} samples=static{len(SAMPLE_STATIC_PROMPTS)}+rand{SAMPLE_RANDOM}@{SAMPLE_TOKENS}tok")
print(f"lr:   matrix={MATRIX_LR} emb={EMBEDDING_LR} unemb={UNEMBEDDING_LR} scalar={SCALAR_LR} warmup={WARMUP_RATIO}")
if major < 8:
    print("note: T4-class GPU - this will be SLOW for a 1.38B model. If you see NaNs in fp16, lower the LR (edit the train cell: add --matrix-lr 0.01).")

## 3. Clone the repo

`Axiom-Inference-Engine` (the `.axim` tooling, SFT trainer, and unified `axim` CLI). Re-running skips an already-cloned dir.

In [ ]:
!test -d {AXIM_REPO}/.git || git clone --depth 1 https://github.com/StrawberCar/Axiom-Inference-Engine.git {AXIM_REPO}

## 4. Install the package

`pip install -e {AXIM_REPO}` installs the `axim` package (CLI + core + SFT trainer) and its dependencies. The console-script `axim` lands on PATH.

In [ ]:
!pip install -e {AXIM_REPO}

Quick CLI sanity check (`axim` is installed and on PATH):

In [ ]:
!axim --help

## 5. Download the base model

`axim download` grabs the ready `.axim` from the Hub. (If a repo ever has no `.axim`, it falls back to safetensors+config+tokenizer and packs one.)

In [ ]:
!axim download --repo {HF_REPO} --output {BASE_AXIM}

In [ ]:
!axim inspect {BASE_AXIM}

## 6. Prepare the dataset

`axim prepare-data` loads the dataset, normalizes any common shape (chat / ShareGPT / Alpaca / OpenMath / question-answer / prompt-completion / raw text) to the nanoChat chat format, and writes train.jsonl + val.jsonl with a stable split.

In [ ]:
## 6. Prepare the dataset (backbone + automated personality blend)
import subprocess, sys

def run_prep(args):
    cmd=["axim", "prepare-data"]+args
    print("$ "+" ".join(cmd))
    subprocess.run(cmd, check=True)

# backbone -> train.jsonl / val.jsonl
run_prep(["--repo", DATASET_REPO]
         + (["--config", DATASET_CONFIG] if DATASET_CONFIG else [])
         + ["--train-examples", str(TRAIN_EXAMPLES),
            "--val-examples", str(VAL_EXAMPLES),
            "--output-train", TRAIN_JSONL, "--output-val", VAL_JSONL,
            "--seed", str(SEED)])

# automated personality blend: append the persona dataset PERSONA_REPEAT times
# onto the train file. val.jsonl stays backbone-only (clean chat-capability signal).
if PERSONA_REPO:
    persona_train = f"{WORK}/persona_train.jsonl"
    run_prep(["--repo", str(PERSONA_REPO)]
             + (["--config", str(PERSONA_CONFIG)] if PERSONA_CONFIG else [])
             + ["--train-examples", "1000", "--val-examples", "0",
                "--output-train", persona_train,
                "--output-val", f"{WORK}/persona_val.jsonl",
                "--seed", str(SEED)])
    with open(persona_train, encoding="utf-8") as pin:
        pdata = pin.read()
    with open(TRAIN_JSONL, "a", encoding="utf-8") as out:
        for _ in range(int(PERSONA_REPEAT)):
            out.write(pdata)
    n_back = sum(1 for _ in open(TRAIN_JSONL, encoding="utf-8"))
    print(f"blended persona {PERSONA_REPO} x{PERSONA_REPEAT} -> train.jsonl now {n_back:,} rows")
else:
    print("PERSONA_REPO is None -> backbone-only training")

# ---- validate/repair conversations so sft_data.render_conversation accepts them ----
# sft_data requires: optional leading system, then strict user/assistant alternation
# starting with user. zerofata character cards often open with an assistant "greeting"
# turn ([system, assistant, user, ...]); drop that leading greeting to repair.
# Any row still invalid after repair is dropped (keeps training crash-free).
import json as _json
NL = chr(10)
def _sft_valid(msgs):
    if not msgs: return False
    i = 0
    if msgs[0].get("role") == "system":
        if len(msgs) < 2 or msgs[1].get("role") != "user": return False
        i = 1
    for j, m in enumerate(msgs[i:]):
        if m.get("role") != ("user" if j % 2 == 0 else "assistant"): return False
    return any(m.get("role") == "assistant" for m in msgs)

def _repair(msgs):
    if len(msgs) >= 2 and msgs[0].get("role") == "system" and msgs[1].get("role") == "assistant":
        msgs = msgs[:1] + msgs[2:]  # drop the leading assistant greeting
    return msgs

for path in (TRAIN_JSONL, VAL_JSONL):
    kept = repaired = dropped = 0
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            conv = _json.loads(line)
            msgs = conv.get("messages")
            if _sft_valid(msgs):
                out.append(conv); kept += 1; continue
            msgs2 = _repair(msgs)
            if _sft_valid(msgs2):
                out.append({"messages": msgs2}); repaired += 1
            else:
                dropped += 1
    with open(path, "w", encoding="utf-8") as f:
        for c in out:
            f.write(_json.dumps(c, ensure_ascii=False) + NL)
    print(f"{path}: kept {kept:,} (repaired {repaired}, dropped {dropped})")

## 7. Fine-tune

Runs `axim sft` with the auto-tuned plan (the `--config` file supplies defaults; every flag below overrides it). Output streams live below. The fine-tuned model is written to `/content/axim_work/finetuned.axim`.

In [ ]:
!axim sft --config {AXIM_REPO}/configs/sft_example.json --base-model {BASE_AXIM} --train {TRAIN_JSONL} --val {VAL_JSONL} --output {OUT_AXIM} --out-dir {OUT_DIR} --device cuda --dtype {DTYPE} --cast-dtype {CAST} --compile {COMPILE} --optimizer {OPT} --batch-size {BATCH} --grad-accum {GRAD_ACCUM} --max-seq-len {SEQ} --epochs {EPOCHS} --train-on {TRAIN_ON} --pack {PACK} --seed {SEED} --eval-every {EVAL_EVERY} --sample-every {SAMPLE_EVERY} --sample-tokens {SAMPLE_TOKENS} --sample-random {SAMPLE_RANDOM} --sample-static-file {SAMPLE_STATIC_FILE} --save-every {SAVE_EVERY} --checkpoint-every -1 --save-optim 0 --log-every {LOG_EVERY} --wandb dummy --matrix-lr {MATRIX_LR} --embedding-lr {EMBEDDING_LR} --unembedding-lr {UNEMBEDDING_LR} --scalar-lr {SCALAR_LR} --warmup-ratio {WARMUP_RATIO} --init-lr-frac {INIT_LR_FRAC} {SYSARG}

In [ ]:
import json
from axim.core import load_axim
d = load_axim(OUT_AXIM)
print("finetuned flag:", d["config"].get("finetuned"))
print(json.dumps(d["metadata"].get("sft", {}), indent=2)[:500])

## 8. Generate from the fine-tuned model

Edit the prompt, then run the inference command.

In [ ]:
PROMPT = "Explain how a transformer handles long-range dependencies, step by step."
SYSTEM_PROMPT = None   # optional, e.g. "You are a concise, helpful assistant."
# MAXTOK is the "diaper": a hard cap on generation. The SFT model is trained to
# emit <|assistant_end|> when done and we stop on it; MAXTOK only kicks in if it
# fails to stop. 256 is a safe default for short chat answers.
MAXTOK = 256
import shlex
PROMPT_Q = shlex.quote(PROMPT)
# empty string when unset -> --system-prompt "" which the script treats as none
SYSTEM_Q = shlex.quote(SYSTEM_PROMPT) if SYSTEM_PROMPT else '""'
print("prompt:", PROMPT)


In [ ]:
!axim infer --axim {OUT_AXIM} --device cuda --prompt {PROMPT_Q} --max-tokens {MAXTOK} --temperature 0.7 --top-k 50 --repetition-penalty 1.1 --chat --system-prompt {SYSTEM_Q}

## 9. Download the result

Pull `/content/axim_work/finetuned.axim` down to your machine.


In [ ]:
from google.colab import files
files.download(OUT_AXIM)